# Cibuco_Boriken — BirdCLEF+ 2026 Inference v2
**Team:** Cibuco_Boriken | **Backbone:** EfficientNet-B2 | **CFAR T=0.3** | **Smart Crop + Secondary Labels** | **234 species**

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Clone repo + add to path ──
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
import sys
sys.path.insert(0, '/kaggle/working/cibuco-boriken')
print('Repo cloned ✅')

In [ ]:
# ── Cell 3: Imports + config ──
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import librosa
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
# ⬇️ UPDATE THIS to your uploaded B2 model dataset path on Kaggle
MODEL_PATH  = Path('/kaggle/input/datasets/drakus74/efficientnet-b2-smartcrop/efficientnet_b2_SMARTCROP_BCE_20260315.pt')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# Config — MUST match training pipeline (birdclef/config.py)
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0
HOP_SEC     = 2.5     # 50% overlap for better boundary coverage
N_MELS      = 128
N_FFT       = 2048    # matches training (was 1024 — WRONG in v1)
HOP_LENGTH  = 512     # matches training (was 320 — WRONG in v1)
FMIN        = 50
FMAX        = 14000
TEMPERATURE = 0.3     # CFAR temperature scaling
BATCH_SIZE  = 64
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# Zero-shot prior: 28 species in taxonomy but not in training data
# Using 1/234 ≈ 0.00427 instead of 0.0 avoids guaranteed AUC=0.5 for these
ZERO_SHOT_PRIOR = 1.0 / 234

print(f'Device:      {DEVICE}')
print(f'Window:      {WINDOW_SEC}s (50% overlap → {HOP_SEC}s hop)')
print(f'Mel:         n_fft={N_FFT}, hop={HOP_LENGTH}, n_mels={N_MELS}')
print(f'Temperature: {TEMPERATURE}')
print(f'Zero-shot prior: {ZERO_SHOT_PRIOR:.6f}')
print('Config ready ✅')

In [ ]:
# ── Cell 4: Load species lists ──
taxonomy_df    = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES        = taxonomy_df['primary_label'].tolist()
NUM_SPECIES    = len(SPECIES)

train_df          = pd.read_csv(DATA_DIR / 'train.csv')
TRAIN_SPECIES     = sorted(train_df['primary_label'].unique().tolist())
NUM_TRAIN_SPECIES = len(TRAIN_SPECIES)

# Zero-shot species: in taxonomy but not in training data
ZERO_SHOT = set(SPECIES) - set(TRAIN_SPECIES)

print(f'Taxonomy species:   {NUM_SPECIES}')
print(f'Train species:      {NUM_TRAIN_SPECIES}')
print(f'Zero-shot species:  {len(ZERO_SHOT)}')
print(f'First 5 zero-shot:  {sorted(ZERO_SHOT)[:5]}')

In [ ]:
# ── Cell 5: Load EfficientNet-B2 model (MUST match training arch) ──
from torchvision.models import efficientnet_b2

# Build EXACTLY the same architecture as birdclef/model.py build_efficientnet_b2()
model = efficientnet_b2(weights=None)
in_features = model.classifier[1].in_features  # 1408 for B2
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, NUM_TRAIN_SPECIES),  # 206 training species
)

# Load trained weights
state_dict = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
# Handle torch.compile _orig_mod prefix
cleaned = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
model.load_state_dict(cleaned)

model = model.to(DEVICE)
model.eval()
print(f'EfficientNet-B2 loaded on {DEVICE} ✅')
print(f'Classifier: {in_features} → {NUM_TRAIN_SPECIES} species')

In [ ]:
# ── Cell 6: Audio + mel utilities (MATCHING training pipeline) ──
def audio_to_mel(audio, sr=SAMPLE_RATE, n_mels=N_MELS):
    """Produce mel-spectrogram with EXACT same params as training."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=N_FFT, hop_length=HOP_LENGTH,       # 2048/512 — matches config.py
        fmin=FMIN, fmax=FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-8:
        mel_db = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_db = np.zeros_like(mel_db)
    return mel_db.astype(np.float32)

def mel_to_tensor(mel):
    """Convert (n_mels, time) → (3, 224, 224) exactly like melspec_to_tensor()."""
    t = torch.from_numpy(mel).unsqueeze(0)          # (1, n_mels, time)
    t = t.repeat(3, 1, 1)                            # (3, n_mels, time)
    t = nnF.interpolate(
        t.unsqueeze(0), size=(224, 224),
        mode='bilinear', align_corners=False,
    ).squeeze(0)                                      # (3, 224, 224)
    return t

def load_audio_windows(filepath, window_sec=WINDOW_SEC, hop_sec=HOP_SEC, sr=SAMPLE_RATE):
    """Load audio with 50% overlap for better boundary coverage."""
    try:
        audio, _ = librosa.load(filepath, sr=sr, mono=True)
    except Exception as e:
        print(f'Error loading {filepath}: {e}')
        return [], []

    window_samples = int(window_sec * sr)
    hop_samples    = int(hop_sec * sr)
    windows, end_times = [], []

    start = 0
    while start + window_samples <= len(audio):
        chunk = audio[start:start + window_samples]
        windows.append(chunk)
        end_times.append(int((start + window_samples) / sr))
        start += hop_samples

    # Last partial window — pad if needed
    if start < len(audio):
        chunk = audio[start:]
        chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        windows.append(chunk)
        end_times.append(int((start + window_samples) / sr))

    return windows, end_times

@torch.no_grad()
def predict_batch(batch_tensor):
    """Run model + temperature-scaled sigmoid."""
    logits = model(batch_tensor)
    return torch.sigmoid(logits / TEMPERATURE).cpu().numpy()

print('Audio utilities ready ✅')

In [ ]:
# ── Cell 7: CFAR adaptive thresholding + overlap pooling ──
def cfar_threshold(all_probs, k=2.0, floor=0.05, ceiling=0.95):
    """Per-species adaptive threshold: mu + k*sigma."""
    mu    = all_probs.mean(axis=0)
    sigma = all_probs.std(axis=0)
    thresholds = np.clip(mu + k * sigma, floor, ceiling)
    return thresholds

def pool_to_grid(end_times, probs, grid_sec=5):
    """Average overlapping window predictions into 5-second grid cells."""
    if len(end_times) == 0:
        return [], np.empty((0, probs.shape[1]))

    max_end = max(end_times)
    num_cells = int(np.ceil(max_end / grid_sec))
    num_species = probs.shape[1]

    accum = np.zeros((num_cells, num_species), dtype=np.float64)
    counts = np.zeros(num_cells, dtype=np.int32)

    for idx, end_t in enumerate(end_times):
        start_t = end_t - WINDOW_SEC
        center = (start_t + end_t) / 2.0
        cell = min(int(center / grid_sec), num_cells - 1)
        accum[cell] += probs[idx]
        counts[cell] += 1

    for c in range(num_cells):
        if counts[c] > 0:
            accum[c] /= counts[c]

    grid_end_times = [(c + 1) * grid_sec for c in range(num_cells)]
    return grid_end_times, accum

print('CFAR + pooling ready ✅')

In [ ]:
# ── Cell 8: Run inference on all test soundscapes ──
soundscape_dir   = DATA_DIR / 'test_soundscapes'
soundscape_files = sorted(soundscape_dir.glob('*.ogg'))
print(f'Test soundscapes: {len(soundscape_files)}')

# Build train_species → index map
train_species_idx = {sp: i for i, sp in enumerate(TRAIN_SPECIES)}

all_rows = []

for sf in tqdm(soundscape_files, desc='Inference'):
    file_id = sf.stem

    windows, end_times = load_audio_windows(str(sf))
    if not windows:
        continue

    # Convert to mel → tensor → (3, 224, 224)
    tensors = []
    for w in windows:
        mel = audio_to_mel(w)
        t = mel_to_tensor(mel)
        tensors.append(t)
    batch_all = torch.stack(tensors).to(DEVICE)   # (N, 3, 224, 224)

    # Batched inference → probs over TRAIN_SPECIES (206)
    probs_list = []
    for i in range(0, len(batch_all), BATCH_SIZE):
        batch = batch_all[i:i+BATCH_SIZE]
        probs_list.append(predict_batch(batch))
    train_probs = np.vstack(probs_list)  # (N_windows, 206)

    # Pool overlapping windows → 5-second grid
    grid_end_times, pooled_probs = pool_to_grid(end_times, train_probs)

    # Build submission rows — map 206 train probs → 234 taxonomy species
    for idx, end_time in enumerate(grid_end_times):
        row_id = f'{file_id}_{end_time}'
        row = {'row_id': row_id}

        for sp in SPECIES:
            if sp in train_species_idx:
                row[sp] = float(pooled_probs[idx, train_species_idx[sp]])
            else:
                # Zero-shot species: use informative prior instead of 0.0
                row[sp] = ZERO_SHOT_PRIOR

        all_rows.append(row)

print(f'Total rows generated: {len(all_rows)} ✅')

In [ ]:
# ── Cell 9: Generate submission.csv ──
sample_sub    = pd.read_csv(DATA_DIR / 'sample_submission.csv')
expected_cols = sample_sub.columns.tolist()

submission_df = pd.DataFrame(all_rows, columns=expected_cols)
submission_df = submission_df.fillna(ZERO_SHOT_PRIOR)  # Fill gaps with prior, not 0
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape}')
print(f'Min prob: {submission_df.iloc[:, 1:].min().min():.6f}')
print(f'Max prob: {submission_df.iloc[:, 1:].max().max():.6f}')
print(submission_df.head(3))

In [ ]:
# ── Cell 10: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Row IDs match:  {set(sample_sub["row_id"]) == set(our_sub["row_id"])}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print(f'No zeros:       {(our_sub.iloc[:, 1:] == 0).sum().sum() == 0}')
print()

if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ SUBMISSION VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    extra   = set(our_sub.columns) - set(sample_sub.columns)
    if missing: print(f'⚠️  Missing columns: {missing}')
    if extra:   print(f'⚠️  Extra columns: {extra}')